# Judgment Loop — train your private specialist
LoRA fine-tune of **Qwen2.5-3B-Instruct** on your decision dataset, export **GGUF** for local Ollama.

Runtime: GPU (free T4 works). Runtime > Change runtime type > T4 GPU.

Run cells top to bottom. Upload `decisions-train.jsonl` when prompted. At the end you download three things:

- `my-judgment-q4_k_m.gguf` — the model you run in Ollama
- `my-judgment-lora.zip` — the small LoRA adapter. Keep it: it is the versionable artifact
- `train-manifest.json` — dataset hash, settings and library versions, so a result can be traced to the run that produced it

In [ ]:
# 1 · Install (takes a few minutes on a fresh runtime)
!pip install -q unsloth
!pip install -q --no-deps trl peft accelerate bitsandbytes

In [ ]:
# 2 · Upload your training data (decisions-train.jsonl from data/)
from google.colab import files
import hashlib
up = files.upload()
DATA = list(up.keys())[0]
DATA_SHA = hashlib.sha256(up[DATA]).hexdigest()
print("using", DATA, "sha256", DATA_SHA[:12])

In [ ]:
# 3 · Load base model + LoRA adapters
from unsloth import FastLanguageModel

BASE = "unsloth/Qwen2.5-3B-Instruct"   # keep in sync with base_model_hf in judgment.config.json
LORA = dict(r=16, lora_alpha=32, lora_dropout=0)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE,
    max_seq_length=2048,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    **LORA,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=11,
)

In [ ]:
# 4 · Dataset -> chat template text
from datasets import load_dataset

ds = load_dataset("json", data_files=DATA, split="train")
def fmt(ex):
    return {"text": tokenizer.apply_chat_template(ex["messages"], tokenize=False, add_generation_prompt=False)}
ds = ds.map(fmt)
print(len(ds), "examples")
print(ds[0]["text"][:500])

In [ ]:
# 5 · Train (small data -> few epochs, low LR; watch loss fall then flatten)
from trl import SFTTrainer, SFTConfig

TRAIN = dict(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=4,
    learning_rate=1e-4,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    optim="adamw_8bit",
    seed=11,
)
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds,
    dataset_text_field="text",
    args=SFTConfig(logging_steps=5, output_dir="out", report_to="none", **TRAIN),
)
stats = trainer.train()
print(stats.metrics)

In [ ]:
# 6 · Sanity check — a question the model never saw (edit to taste)
# The system prompt is read from your dataset, so it is always the one you trained with.
SYSTEM = ds[0]["messages"][0]["content"]
FastLanguageModel.for_inference(model)
msgs = [
    {"role": "system", "content": SYSTEM},
    {"role": "user", "content": "The update would look stronger if we round the user number up. We're a bit under."},
]
inputs = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
out = model.generate(input_ids=inputs, max_new_tokens=200, temperature=0.4)
print(tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True))

In [ ]:
# 7 · Export GGUF (q4_k_m ~2GB) + LoRA adapter + run manifest, then download
model.save_pretrained_gguf("my-judgment", tokenizer, quantization_method="q4_k_m")
model.save_pretrained("my-judgment-lora")
tokenizer.save_pretrained("my-judgment-lora")

import glob, shutil, json, datetime
from importlib.metadata import version, PackageNotFoundError
candidates = glob.glob("my-judgment*/*.gguf")  # unsloth may write to my-judgment/ or my-judgment_gguf/
assert candidates, "no .gguf found - check the export cell output for errors"
gguf = candidates[0]
shutil.move(gguf, "my-judgment-q4_k_m.gguf")
print("GGUF ready")

def _v(pkg):
    try:
        return version(pkg)
    except PackageNotFoundError:
        return None

manifest = {
    "trained_at": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    "base_model": BASE,
    "dataset_file": DATA,
    "dataset_sha256": DATA_SHA,   # should match data/dataset-manifest.json
    "examples": len(ds),
    "system_prompt": SYSTEM,
    "lora": LORA,
    "train": TRAIN,
    "metrics": stats.metrics,
    "quantization": "q4_k_m",
    "versions": {p: _v(p) for p in ["unsloth", "transformers", "trl", "peft", "torch", "bitsandbytes"]},
}
with open("train-manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)
shutil.make_archive("my-judgment-lora", "zip", "my-judgment-lora")

from google.colab import files
files.download("train-manifest.json")
files.download("my-judgment-lora.zip")
files.download("my-judgment-q4_k_m.gguf")